In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent = create_agent(
    model = 'groq:llama-3.1-8b-instant',
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = 'groq:whisper-large-v3-turbo',
            trigger = ("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [7]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-1"}}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='89f64001-d7e5-4f55-ae1a-c06f45976ab0'), AIMessage(content='The answer to 2 + 2 is 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 42, 'total_tokens': 55, 'completion_time': 0.01712299, 'completion_tokens_details': None, 'prompt_time': 0.002969021, 'prompt_tokens_details': None, 'queue_time': 0.045903289, 'total_time': 0.020092011}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd904-c6ab-7560-a35b-f2273372b8b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 13, 'total_tokens': 55})]}
Messages:2
Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='89f64001-d7e5-4f55-ae1a-c06f45976ab0

#### HUMAN IN LOOP - MIDDLEWARE

In [8]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [10]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
agent=create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [16]:
config = {"configurable": {"thread_id": "test-approve"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [19]:
from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages']}")

 Paused! Approving...
Result: [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='517fd998-afc9-4bf3-b41f-32e900bb9bd1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ksesxjhd4', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.034876285, 'completion_tokens_details': None, 'prompt_time': 0.021096716, 'prompt_tokens_details': None, 'queue_time': 0.046172353, 'total_time': 0.055973001}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd90d-4d77-7533-8132-5f436040e27d-0', tool_calls=[{'name': 'send_email_tool',

In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='517fd998-afc9-4bf3-b41f-32e900bb9bd1'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ksesxjhd4', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.034876285, 'completion_tokens_details': None, 'prompt_time': 0.021096716, 'prompt_tokens_details': None, 'queue_time': 0.046172353, 'total_time': 0.055973001}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd90d-4d77-7533-8132-5f436040e27d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body